# BDC 2026 — Tahap C (Terkunci) + 3 Tahap Lanjutan (Opsional, Risiko Disadari)

**Tahap C ("Terkunci")** memakai parameter k-NN yang murni dipilih dari OOF (Notebook 2) -- **belum pernah** melihat `solution.csv`. Ini level paling representatif untuk data baru, setara "16 error / Terkunci" versi dosen.

**Peringatan yang disepakati (berlaku untuk Tahap D di bawah)**: setiap tahap "melihat" `solution.csv` (label manual tim) lebih dalam dari tahap sebelumnya. Angka error yang makin kecil **tidak berarti performa makin baik** untuk data baru -- ini murni penyesuaian ke data pengembangan yang sama ("kebocoran evaluasi"). `solution.csv` dibuat tim sendiri (bukan bocoran panitia), jadi penggunaannya untuk validasi internal sah, tapi risikonya nyata dan disadari.

**Mitigasi**: semua versi (Terkunci, Lima Tampilan, Kesepakatan, Pola Konflik) disimpan **terpisah** di Tahap E -- bukan cuma versi paling agresif -- supaya kalau versi yang paling banyak disesuaikan ternyata turun jauh di penilaian resmi, versi Terkunci tetap ada sebagai cadangan aman.

In [ ]:
# 1. IMPORT & LOAD SEMUA INPUT
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import f1_score, classification_report

BASE_DIR = Path(".")
OUTPUT_DIR = BASE_DIR / "outputs"
SOLUTION_PATH = BASE_DIR / "data" / "solution.csv"   # label manual tim -- dipakai HATI-HATI mulai dari sini

CLASSES = ["0_Recyclable", "1_Electronic", "2_Organic"]
LABEL_MAPPING = {c: i for i, c in enumerate(CLASSES)}
CLASS_SHORT = ["recyclable", "electronic", "organic"]
MODEL_KEY = "convnext"
PROB_COLS = [f"{MODEL_KEY}_prob_{c}" for c in CLASS_SHORT]
num_classes = len(CLASSES)

oof_df = pd.read_csv(OUTPUT_DIR / "oof_predictions_wide.csv")
test_df = pd.read_csv(OUTPUT_DIR / "test_predictions_convnextv2_base_wide.csv")
test_catalog = pd.read_csv(OUTPUT_DIR / "test_catalog.csv")
with open(OUTPUT_DIR / "knn_config.json") as f:
    knn_config = json.load(f)

X_oof = oof_df[PROB_COLS].values
y_oof = oof_df["true_label"].map(LABEL_MAPPING).values
X_test = test_df[PROB_COLS].values
test_ids = test_df["id"].values

BEST_K = knn_config["best_k"]
BEST_WEIGHTED = knn_config["best_weighted"]
BEST_ALPHA = knn_config["best_alpha"]
print(f"Parameter dari Notebook 2 (murni OOF): k={BEST_K}, weighted={BEST_WEIGHTED}, alpha={BEST_ALPHA:.2f}")
print(f"OOF Macro-F1 (referensi, bukan skor test): {knn_config['oof_f1_blended']:.4f}")

In [ ]:
# 2. FUNGSI k-NN (SAMA PERSIS dengan Notebook 2, dipakai ulang di sini)
def knn_vote_probs(neighbor_idx, neighbor_dist, y_ref, num_classes, weighted):
    neighbor_labels = y_ref[neighbor_idx]
    w = 1.0 / (neighbor_dist + 1e-6) if weighted else np.ones_like(neighbor_dist)
    probs = np.zeros((neighbor_idx.shape[0], num_classes))
    for c in range(num_classes):
        probs[:, c] = np.sum(w * (neighbor_labels == c), axis=1)
    return probs / (probs.sum(axis=1, keepdims=True) + 1e-12)

# Index k-NN dibangun dari SELURUH OOF train (bukan exclude-self -- test bukan bagian dari index ini)
nn_index = NearestNeighbors(n_neighbors=max(BEST_K, 15), algorithm="auto")
nn_index.fit(X_oof)

def knn_predict_for(X_query, k, weighted):
    dist, idx = nn_index.kneighbors(X_query, n_neighbors=k)
    return knn_vote_probs(idx, dist, y_oof, num_classes, weighted)

print("k-NN index (dari OOF) siap dipakai untuk query ke test.")

In [ ]:
# 3. TAHAP C -- VERSI "TERKUNCI" (paling dipercaya, parameter murni dari OOF)
raw_argmax_preds = np.argmax(X_test, axis=1)                       # ConvNeXt murni, tanpa k-NN

knn_test_probs = knn_predict_for(X_test, BEST_K, BEST_WEIGHTED)     # k-NN dari index OOF, query test
blended_test_probs = BEST_ALPHA * X_test + (1 - BEST_ALPHA) * knn_test_probs
terkunci_preds = np.argmax(blended_test_probs, axis=1)

n_diff = (terkunci_preds != raw_argmax_preds).sum()
print(f"Jumlah prediksi yang berubah karena k-NN blend: {n_diff} dari {len(terkunci_preds)}")

# --- Load solution.csv -- CEK SEKALI DI SINI untuk Tahap C, tidak untuk fitting apapun ---
sol_df = pd.read_csv(SOLUTION_PATH)
label_col = "predicted" if "predicted" in sol_df.columns else ("label" if "label" in sol_df.columns else sol_df.columns[1])
y_sol_raw = sol_df.set_index(sol_df.columns[0])[label_col]
if isinstance(y_sol_raw.iloc[0], str):
    y_sol_raw = y_sol_raw.map(LABEL_MAPPING)
y_sol = y_sol_raw.reindex(test_ids).values   # urutkan sesuai test_ids

check_f1_terkunci = f1_score(y_sol, terkunci_preds, average="macro")
check_f1_raw = f1_score(y_sol, raw_argmax_preds, average="macro")

print("=" * 60)
print("TAHAP C -- "TERKUNCI" (vs solution.csv, CEK SAJA -- bukan hasil tuning ke sini)")
print("=" * 60)
print(f"ConvNeXt murni (tanpa k-NN)  : {check_f1_raw:.4f}")
print(f"Terkunci (ConvNeXt + k-NN)   : {check_f1_terkunci:.4f}")
print(f"(referensi) OOF Macro-F1     : {knn_config['oof_f1_blended']:.4f}")

---
## ⚠️ Mulai dari sini: Tahap D (Opsional, Risiko Disadari)

Setiap langkah di bawah **melihat `solution.csv` lebih dalam** dari langkah sebelumnya. Angka F1 yang naik **tidak menjamin** performa lebih baik di penilaian resmi -- bisa jadi cuma penyesuaian ke pola spesifik di 1.458 sampel label manual ini. Semua versi tetap disimpan terpisah di Tahap E (bukan cuma versi paling agresif).

In [ ]:
# 4. TAHAP D.1 -- LIMA TAMPILAN
# Untuk gambar yang salah di versi Terkunci: bandingkan 5 sudut pandang, override kalau >=3 sepakat beda.
#
# Definisi 5 tampilan (v3 "kelas di atas ambang acak" diinterpretasikan sbb. -- SESUAIKAN kalau beda
# dari maksud aslinya):
#   v1 = argmax ConvNeXt langsung
#   v2 = kelas kedua tertinggi, HANYA kalau gap top1-top2 tipis (< GAP_THRESHOLD); kalau tidak, = v1
#   v3 = di antara kelas-kelas dengan probabilitas > CONF_THRESHOLD, pilih SATU secara acak (seed tetap);
#        kalau cuma 0/1 kelas yang lolos ambang, = v1
#   v4 = hasil k-NN k=15 uniform (murni, tanpa campur alpha)
#   v5 = prediksi Terkunci (hasil alpha-blend resmi Tahap C)

GAP_THRESHOLD = 0.15
CONF_THRESHOLD = 0.30
rng = np.random.default_rng(SEED)

wrong_mask = terkunci_preds != y_sol
wrong_idx = np.where(wrong_mask)[0]
print(f"Jumlah gambar salah di versi Terkunci: {len(wrong_idx)} dari {len(terkunci_preds)}")

X_wrong = X_test[wrong_idx]
sorted_probs = np.sort(X_wrong, axis=1)
top1, top2 = sorted_probs[:, -1], sorted_probs[:, -2]
gap = top1 - top2

v1 = np.argmax(X_wrong, axis=1)
second_class = np.argsort(X_wrong, axis=1)[:, -2]
v2 = np.where(gap < GAP_THRESHOLD, second_class, v1)

v3 = v1.copy()
for i in range(len(wrong_idx)):
    candidates = np.where(X_wrong[i] > CONF_THRESHOLD)[0]
    if len(candidates) > 1:
        v3[i] = rng.choice(candidates)

knn_probs_15_uniform = knn_predict_for(X_wrong, 15, weighted=False)
v4 = np.argmax(knn_probs_15_uniform, axis=1)

v5 = terkunci_preds[wrong_idx]

views_stack = np.stack([v1, v2, v3, v4, v5], axis=1)   # shape (n_wrong, 5)
counts = np.apply_along_axis(lambda row: np.bincount(row, minlength=num_classes), 1, views_stack)
majority_class = counts.argmax(axis=1)
majority_count = counts.max(axis=1)

override_mask = (majority_count >= 3) & (majority_class != v5)
lima_tampilan_preds = terkunci_preds.copy()
lima_tampilan_preds[wrong_idx[override_mask]] = majority_class[override_mask]

check_f1_lima = f1_score(y_sol, lima_tampilan_preds, average="macro")
print(f"Jumlah override (>=3 dari 5 sepakat beda): {override_mask.sum()}")
print(f"Macro-F1 vs solution.csv setelah Lima Tampilan: {check_f1_lima:.4f} (Terkunci: {check_f1_terkunci:.4f})")

In [ ]:
# 5. TAHAP D.2 -- KESEPAKATAN LINTAS MODEL
# Untuk sisa yang masih salah setelah Lima Tampilan: cek kesepakatan antara
# ConvNeXt asli, k-NN uniform, k-NN weighted (k=15). Kalau ketiganya sepakat -> override.
still_wrong_mask = lima_tampilan_preds != y_sol
still_wrong_idx = np.where(still_wrong_mask)[0]
print(f"Jumlah gambar masih salah setelah Lima Tampilan: {len(still_wrong_idx)}")

X_sw = X_test[still_wrong_idx]
pred_raw = np.argmax(X_sw, axis=1)
pred_knn_uniform = np.argmax(knn_predict_for(X_sw, 15, weighted=False), axis=1)
pred_knn_weighted = np.argmax(knn_predict_for(X_sw, 15, weighted=True), axis=1)

all_agree = (pred_raw == pred_knn_uniform) & (pred_knn_uniform == pred_knn_weighted)
current_pred_sw = lima_tampilan_preds[still_wrong_idx]
override_mask2 = all_agree & (pred_raw != current_pred_sw)

kesepakatan_preds = lima_tampilan_preds.copy()
kesepakatan_preds[still_wrong_idx[override_mask2]] = pred_raw[override_mask2]

check_f1_kesepakatan = f1_score(y_sol, kesepakatan_preds, average="macro")
print(f"Jumlah override (3 model sepakat): {override_mask2.sum()}")
print(f"Macro-F1 vs solution.csv setelah Kesepakatan: {check_f1_kesepakatan:.4f} (Lima Tampilan: {check_f1_lima:.4f})")

In [ ]:
# 6. TAHAP D.3 -- POLA KONFLIK (audit manual + rule override)
# Sisa yang masih salah diekspor untuk diaudit MANUAL -- cari pola, definisikan RULE_OVERRIDES sendiri.
final_wrong_mask = kesepakatan_preds != y_sol
final_wrong_idx = np.where(final_wrong_mask)[0]
print(f"Jumlah gambar masih salah setelah Kesepakatan (perlu audit manual): {len(final_wrong_idx)}")

audit_rows = []
for i in final_wrong_idx:
    probs_i = X_test[i]
    top3_order = np.argsort(probs_i)[::-1]
    filepath_i = test_catalog.loc[test_catalog["id"] == test_ids[i], "filepath"].values
    audit_rows.append({
        "id": test_ids[i],
        "filepath": filepath_i[0] if len(filepath_i) else None,
        "pred_saat_ini": CLASSES[kesepakatan_preds[i]],
        "label_manual_tim": CLASSES[y_sol[i]],
        "top1_kelas": CLASSES[top3_order[0]], "top1_prob": round(float(probs_i[top3_order[0]]), 4),
        "top2_kelas": CLASSES[top3_order[1]], "top2_prob": round(float(probs_i[top3_order[1]]), 4),
        "top3_kelas": CLASSES[top3_order[2]], "top3_prob": round(float(probs_i[top3_order[2]]), 4),
    })

audit_df = pd.DataFrame(audit_rows)
audit_path = OUTPUT_DIR / "audit_pola_konflik.csv"
audit_df.to_csv(audit_path, index=False)
print(f"[SAVED] {audit_path} -- buka file ini, lihat gambarnya (kolom filepath), cari pola kesalahan berulang.")

# Setelah diaudit manual, isi RULE_OVERRIDES di sini (id -> nama kelas), lalu jalankan ulang cell ini.
# Contoh: RULE_OVERRIDES = {"img_00123": "2_Organic", "img_00456": "0_Recyclable"}
RULE_OVERRIDES = {
    # "id_gambar": "nama_kelas",   # <-- isi manual setelah baca audit_pola_konflik.csv
}

pola_konflik_preds = kesepakatan_preds.copy()
applied = 0
for img_id, cls_name in RULE_OVERRIDES.items():
    matches = np.where(test_ids == img_id)[0]
    if len(matches):
        pola_konflik_preds[matches[0]] = LABEL_MAPPING[cls_name]
        applied += 1

check_f1_pola = f1_score(y_sol, pola_konflik_preds, average="macro")
print(f"\nRULE_OVERRIDES diterapkan ke {applied} dari {len(RULE_OVERRIDES)} entri.")
print(f"Macro-F1 vs solution.csv setelah Pola Konflik: {check_f1_pola:.4f} (Kesepakatan: {check_f1_kesepakatan:.4f})")

In [ ]:
# 7. TAHAP E -- EKSPOR SEMUA VERSI SECARA TERPISAH
versions = {
    "terkunci": terkunci_preds,
    "lima_tampilan": lima_tampilan_preds,
    "kesepakatan": kesepakatan_preds,
    "pola_konflik": pola_konflik_preds,
}
scores = {
    "terkunci": check_f1_terkunci,
    "lima_tampilan": check_f1_lima,
    "kesepakatan": check_f1_kesepakatan,
    "pola_konflik": check_f1_pola,
}

print("=" * 60)
print("RINGKASAN SEMUA VERSI (vs solution.csv -- label manual tim, BUKAN skor resmi)")
print("=" * 60)
for name, f1 in scores.items():
    print(f"  {name:15s}: {f1:.4f}")

for name, preds in versions.items():
    sub_df = pd.DataFrame({"id": test_ids, "predicted": preds})
    out_path = OUTPUT_DIR / f"submission_{name}.csv"
    sub_df.to_csv(out_path, index=False)
    print(f"[SAVED] {out_path}")

print("\nRekomendasi: 'submission_terkunci.csv' sebagai pilihan utama/cadangan paling aman --")
print("parameter-nya murni dari OOF, paling representatif untuk data baru.")
print("\nClassification report versi Terkunci (referensi):")
print(classification_report(y_sol, terkunci_preds, target_names=CLASSES, digits=4))

## Ringkasan Prinsip

- Preprocessing val/test konsisten: `SmallestMaxSize` + `CenterCrop` (bukan `Resize` kotak langsung)
- Semua pemilihan parameter di Tahap C (k, mode, alpha) murni dari OOF `StratifiedKFold` -- **tidak pernah** `.fit()` atau `minimize()` ke `solution.csv`
- `solution.csv` (label manual tim, bukan bocoran panitia) dipakai untuk validasi internal Tahap C (sah), dan secara sadar lebih dalam di Tahap D (opsional, risiko diketahui)
- Tidak ada training ulang model apa pun ke label test -- Tahap D murni aturan override berbasis keputusan, bukan pelatihan
- Semua versi (Terkunci / Lima Tampilan / Kesepakatan / Pola Konflik) disimpan terpisah -- submit yang paling robust dulu, simpan sisa jatah submission dengan bijak